In [1]:
# install required libraries
!pip install pandas numpy scikit-learn matplotlib seaborn
!pip install torch transformers datasets evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00


In [5]:
# load dataset
import pandas as pd

df = pd.read_csv("/content/sample_data/IMDB Dataset.csv")
print(df.head())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [6]:
# data cleaning （to remove noise and make vocab smaller)
import re #regex library

def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)   # remove html
    text = re.sub(r'[^a-zA-Z ]', '', text) # remove chars other than aA-zZ and space
    return text

df["clean_review"] = df["review"].apply(clean_text)

In [7]:
print(df["clean_review"].head())

0    one of the other reviewers has mentioned that ...
1    a wonderful little production the filming tech...
2    i thought this was a wonderful way to spend ti...
3    basically theres a family where a little boy j...
4    petter matteis love in the time of money is a ...
Name: clean_review, dtype: object


In [8]:
# train test
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["clean_review"],  # input for x
    df["sentiment"],  # target for y
    test_size=0.2,
    random_state=42
)

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

# use tfidf to vectorize word to number, with importance of each word
vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [10]:
# train classifier
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train_tfidf, y_train) # train the model using x_tfidf (vectorized)

LogisticRegression()

In [11]:
# test the model
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.8927
              precision    recall  f1-score   support

    negative       0.90      0.88      0.89      4961
    positive       0.88      0.91      0.89      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000



In [12]:
# try upgrading to BERT
# MAP label into numbers
label_map = {"negative": 0, "positive": 1}

y_train = y_train.map(label_map)
y_test = y_test.map(label_map)

In [13]:
# create dataframe
train_df = pd.DataFrame({"text": X_train, "label": y_train})
test_df = pd.DataFrame({"text": X_test, "label": y_test})

In [14]:
# convert to huggingface dataset, huggingface datasets are optimized for nlp pipeline
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [15]:
# load bert tokenizer (tokenize sentences into individual words and finally convert to numbers)
from transformers import AutoTokenizer

# bert base uncased = convert all to lowercase
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [16]:
# define tokenization function
def tokenize_function(example):
    return tokenizer( # convert text into input id and attention mask
        example["text"], # get the text column value
        padding="max_length", # token max length. if a sentence<256 = PADDING, else = cut because NN need same size input
        truncation=True, # to cut sentence if exceeeds 256
        max_length=256
    )

train_dataset = train_dataset.map(tokenize_function, batched=True) # batched true means process many sentences altogether to make it faster
test_dataset = test_dataset.map(tokenize_function, batched=True)

# before and after BERT tokenization
# before :
# {
#   "text": "I love NLP",
#   "label": 1
# }

# after
# {
#   "text": "I love NLP",
#   "label": 1,
#   "input_ids": [...256 numbers...],
#   "attention_mask": [...256 numbers...]
# }

Map:   0%|          | 0/40000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [17]:
# set format for pytorch
# this will convert dataset in PyTorch format, and only keep the columns needed for training
# BERT model works well with torch format
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

In [18]:
# load BERT model for classification
from transformers import AutoModelForSequenceClassification
#  AutoModel.... is a helper class that loads a model designed to classify text (like positive vs negative, spam vs not spam, etc.).
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased", # Load the pre-trained BERT model that ignores upper/lowercase
    num_labels=2 # tell the model to classify text into 2 categories
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [19]:
# define training args (training rules/settings for model)
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch", # eval after each epoch
    save_strategy="epoch", # save after each epoch
    learning_rate=2e-5, # learning rate 0.00002, small rate avoid breaking the model
    per_device_train_batch_size=16, # number of training samples processed at once
    per_device_eval_batch_size=16, # number of samples processed at once during eval
    num_train_epochs=2,
    weight_decay=0.01, # prevent overfitting by penalizing very large weight
    logging_dir="./logs",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [20]:
# define Trainer that will execute the training settings
# Trainer will do the model training process for us
from transformers import Trainer
import evaluate  # library to handle all kind of evaluation
import numpy as np # library to deal with numbers and arrays

metric = evaluate.load("accuracy") # use accuracy as the evaluation metric

# below function will be used when the model is evaluated
# logits are raw scores prediction from the BERT model
# labels = correct answers
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1) # convert raw scores to predictions using argmax
    return metric.compute(predictions=predictions, references=labels) # compute accuracy

trainer = Trainer( # set up the trainer
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [21]:
# begin the training process using Trainer (1h04m of processing time)
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.221633,0.237463,0.914900
2,0.134598,0.265015,0.925300


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=5000, training_loss=0.20112268676757813, metrics={'train_runtime': 3849.1255, 'train_samples_per_second': 20.784, 'train_steps_per_second': 1.299, 'total_flos': 1.05244422144e+16, 'train_loss': 0.20112268676757813, 'epoch': 2.0})

In [22]:
# evaluate performance
trainer.evaluate()

{'eval_loss': 0.2650153934955597,
 'eval_accuracy': 0.9253,
 'eval_runtime': 137.843,
 'eval_samples_per_second': 72.546,
 'eval_steps_per_second': 4.534,
 'epoch': 2.0}